# NoseKnows — Synthetic Dataset Generation

**Phase 1:** Preprocessing (CPU, ~5 min) — filter, deduplicate, tier, assign question types  
**Phase 2:** Generation (GPU, multi-session) — Qwen3-8B with capped thinking, checkpoint per batch  

**Resume behaviour:** re-run all cells. Preprocessing re-runs (fast). Generation reads the checkpoint and skips already-processed rows automatically.  

**Outputs** (all in `/kaggle/working/`):  
- `dataset.jsonl` — training examples in messages format  
- `failures.jsonl` — batches Qwen failed to produce valid JSON for (retry manually)  
- `checkpoint.json` — last processed index (do not delete between sessions)  
- `generation_report.json` — final statistics  

**Hardware:** P100 (single GPU). Do not use dual-T4.

---
**Fixes applied vs v1:**  
1. `generate_batch` returns `(results, raw_output)` — raw output always preserved for failure logging  
2. Qwen3 special tokens stripped explicitly after decode  
3. `record_to_dict` uses safe column existence checks instead of `pd.Series.get()`  
4. Boolean mask parentheses made explicit  
5. `tqdm` progress bar added to generation loop  
6. Cell 9 reads all values from disk — safe to run standalone mid-session

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "bitsandbytes==0.46.1",
        "accelerate>=0.29.0",
        "transformers>=4.45.0",
        "tqdm",
    ],
    check=True,
)

# Verify the install
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)
print("Dependencies installed.")

In [ ]:
# ── Cell 2: imports ───────────────────────────────────────────────────────
import gc
import json
import logging
import os
import re
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
log = logging.getLogger("nosknows")

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: constants — all tunable parameters in one place ───────────────

# ── paths ──
INPUT_CSV       = Path("/kaggle/input/datasets/olgagmiufana1/fragrantica-com-fragrance-dataset/fra_cleaned.csv")
OUTPUT_DIR      = Path("/kaggle/working")
DATASET_PATH    = OUTPUT_DIR / "dataset.jsonl"
FAILURES_PATH   = OUTPUT_DIR / "failures.jsonl"
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.json"
REPORT_PATH     = OUTPUT_DIR / "generation_report.json"

# ── model ──
MODEL_ID = "Qwen/Qwen3-8B"
# HF_TOKEN is read from Kaggle secrets at runtime — never hardcode it.

# ── preprocessing thresholds ──
MIN_RATING_COUNT         = 50
MIN_TOTAL_DISTINCT_NOTES = 4
MIN_ACCORDS              = 2
MIN_TOP_NOTES            = 2
YEAR_MIN                 = 1900
YEAR_MAX                 = 2026
# Two perfumes from the same brand differing by <= this many notes are duplicates.
MAX_NOTE_DIFF            = 2
# rating_count >= GOLD_THRESHOLD -> 2 training examples; else 1.
GOLD_THRESHOLD           = 200

# ── generation ──
BATCH_SIZE      = 5      # perfumes per Qwen call
TEMPERATURE     = 0.6
TOP_P           = 0.9
# Covers thinking trace (~512 tokens) + JSON output (~1000 tokens) comfortably.
MAX_NEW_TOKENS  = 2056
# Caps Qwen3's internal reasoning trace. Reduces generation time ~60% vs
# unbounded thinking while preserving most quality. If apply_chat_template
# raises TypeError for this arg the fallback in generate_batch() handles it.
THINKING_BUDGET = 512

# FIX 2: Qwen3 special tokens that survive skip_special_tokens=False decoding.
# Stripped explicitly in extract_json_from_response to keep the output clean.
QWEN_SPECIAL_TOKENS: list[str] = [
    "<|im_end|>",
    "<|endoftext|>",
    "<|im_start|>",
]

# ── question type distribution (must sum to 1.0) ──
QUESTION_TYPES: list[tuple[str, float]] = [
    ("occasion_based",        0.25),
    ("mood_based",            0.25),
    ("note_based",            0.20),
    ("comparison",            0.15),
    ("structured_preference", 0.15),
]

# ── NoseKnows system prompt ──
# Injected programmatically into every training example — never generated by Qwen.
# Byte-for-byte identical across all examples so the fine-tuned head always
# sees the same persona context.
NOSKNOWS_SYSTEM_PROMPT: str = (
    "You are NoseKnows, a fragrance consultant who knows perfumery inside out. "
    "When someone describes what they are after, whether a mood, an occasion, "
    "or notes they love or cannot stand, you recommend real perfumes by name and "
    "brand and explain exactly why they fit, grounding your answer in the actual "
    "notes and accords. Warm, confident, specific. Never vague, never a catalogue. "
    "3 to 5 sentences."
)

# ── Qwen3-8B generator system prompt ──
# Instructs Qwen to act as a data factory, not a chat assistant.
# Few-shot examples cover mood_based, note_based, structured_preference.
# occasion_based and comparison are intentionally left out so Qwen generalises
# to those types rather than copying a template.
GENERATOR_SYSTEM_PROMPT: str = """You are building a training dataset for NoseKnows, an AI fragrance assistant. Your job is writing realistic conversations a real person might have with that assistant.

For each perfume record you receive, you will be told how many examples to write and what question type each one should be. Before writing each example, think about who would actually send this message, what the notes in the record tell you about why this perfume fits, and how someone who genuinely knows fragrance would explain that connection out loud.

The assistant turn must mention the perfume name, the brand, and at least two or three notes pulled directly from the record. It should read like it came from a person who knows what they are talking about, not a product description with names filled in. Warm and direct, 3 to 5 sentences.

Do not invent notes, accords or perfumer names not present in the record. Do not pad. Each answer should feel written for that specific perfume, not recycled from a template.

Output a single valid JSON array. Each object has two fields: "user" and "assistant". The value of both fields is plain prose, not nested JSON. No preamble, no markdown, no explanation. After your thinking, output only a JSON array starting with [ and ending with ]. No text before the [, no text after the ], no markdown fences.

Question types you will see: occasion_based (a specific event or situation), mood_based (a feeling or atmosphere the user wants to evoke), note_based (the user names specific notes or accords they want), comparison (they reference a known perfume and want something similar or different), structured_preference (they list notes they like and notes they want to avoid).

Three examples to calibrate the quality and style expected. Do not copy them:

Example 1 — mood_based:
{"user": "I want something that feels like a quiet evening by the fireplace, maybe a little mysterious.", "assistant": "Shalimar by Guerlain was basically made for that mood. It opens with bergamot, then softens into iris and jasmine before settling into a long base of vanilla, opoponax and smoky incense. There is something almost cinematic about how it develops on skin, intimate and unhurried. A timeless choice for exactly that kind of evening."}

Example 2 — note_based:
{"user": "I keep coming back to oud and leather. Something that really leans into those, not just a hint of them.", "assistant": "Knize Ten by Knize is worth trying if you want those notes front and center. Leather runs through the whole thing, dry and a little animalic, with oud deepening the base alongside wood and a touch of floral that keeps it from getting too heavy. It was created in 1924 and it still smells like nothing else. Not subtle, but that is clearly not what you are after."}

Example 3 — structured_preference:
{"user": "I love amber and sandalwood but anything aquatic or fresh just does not work on me. What would you suggest?", "assistant": "Ambre Sultan by Serge Lutens was designed for people who want amber done properly. The opening is herbal and resinous, coriander and bay leaf, before it moves into a deep heart of pure amber built on sandalwood and benzoin. Nothing fresh, nothing light. It sits close to the skin for hours and the dry-down is genuinely beautiful."}"""

print("Constants set.")
print(f"Input CSV      : {INPUT_CSV}")
print(f"Model          : {MODEL_ID}")
print(f"Batch size     : {BATCH_SIZE} perfumes/call")
print(f"Thinking budget: {THINKING_BUDGET} tokens")
print(f"Temperature    : {TEMPERATURE}")

In [ ]:
# ── Cell 4: preprocessing pipeline ───────────────────────────────────────
# CPU only. ~5 minutes. Fully deterministic — safe to re-run on session resume.

# ── 4.1 normalization helpers ──

def parse_note_field(raw: Any) -> list[str]:
    """
    Parse a raw note string into a clean list of note name strings.
    Strips parenthetical qualifiers ('bergamot (Italian)' -> 'bergamot'),
    unifies ' and ' separators to commas, lowercases everything.
    Returns [] for NaN or empty input.
    """
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    text = re.sub(r"\([^)]*\)", "", str(raw))
    text = re.sub(r"\s+and\s+", ",", text, flags=re.IGNORECASE)
    notes = [n.strip().lower() for n in text.split(",") if n.strip()]
    return [n for n in notes if len(n) > 1 and not n.isdigit()]


def parse_accords(row: pd.Series) -> list[str]:
    """Collect mainaccord1-5 columns into a cleaned list, skipping nulls."""
    accords = []
    for col in ["mainaccord1", "mainaccord2", "mainaccord3", "mainaccord4", "mainaccord5"]:
        if col in row.index and not pd.isna(row[col]):
            val = str(row[col]).strip().lower()
            if val:
                accords.append(val)
    return accords


def normalize_rating_value(raw: Any) -> float:
    """Convert comma-decimal strings ('1,42') to floats (1.42)."""
    if pd.isna(raw):
        return float("nan")
    return float(str(raw).replace(",", "."))


def normalize_gender(raw: Any) -> str:
    """Unify gender variants to 'men' | 'women' | 'unisex' | ''."""
    if pd.isna(raw):
        return ""
    val = str(raw).strip().lower()
    if val in {"men", "male", "for men", "homme"}:
        return "men"
    if val in {"women", "female", "for women", "femme"}:
        return "women"
    if val in {"unisex", "unisex / shared", "shared"}:
        return "unisex"
    return val


def normalize_text(raw: Any) -> str:
    """Strip and collapse internal whitespace. Returns '' for NaN."""
    if pd.isna(raw):
        return ""
    return re.sub(r"\s+", " ", str(raw).strip())


# ── 4.2 load ──
log.info("Loading CSV...")
df = pd.read_csv(INPUT_CSV, sep=";", encoding="latin-1", on_bad_lines="skip")
log.info("Loaded %d rows, %d columns.", len(df), len(df.columns))
print("Columns:", list(df.columns))

stats: dict[str, Any] = {"input_rows": len(df)}

# ── 4.3 structural drop ──
before = len(df)
df = df.dropna(subset=["Perfume", "Brand"])

# FIX 4: explicit parentheses around each side of the | operator.
# & binds tighter than | in Python — without parens the grouping is
# correct by accident but misleading to read and fragile to modify.
has_notes = (
    (df["Top"].notna()  & (df["Top"].str.strip()  != "")) |
    (df["Base"].notna() & (df["Base"].str.strip() != ""))
)
df = df[has_notes]
df = df.dropna(subset=["Rating Count"])
log.info("Structural drop: %d -> %d rows.", before, len(df))
stats["after_structural_drop"] = len(df)

# ── 4.4 normalize ──
log.info("Normalizing fields...")
df["top_notes"]    = df["Top"].apply(parse_note_field)
df["middle_notes"] = df["Middle"].apply(parse_note_field)
df["base_notes"]   = df["Base"].apply(parse_note_field)
df["accords"]      = df.apply(parse_accords, axis=1)
df["perfume_clean"]= df["Perfume"].apply(normalize_text)
df["brand_clean"]  = df["Brand"].apply(normalize_text)
df["gender_clean"] = df["Gender"].apply(normalize_gender)
df["country_clean"]= df["Country"].apply(normalize_text)
df["rating_value"] = df["Rating Value"].apply(normalize_rating_value)
df["rating_count"] = pd.to_numeric(df["Rating Count"], errors="coerce")
df["year"]         = pd.to_numeric(df["Year"], errors="coerce")

# Perfumer columns are optional — produce empty strings if absent in the CSV.
for src_col, dst_col in [
    ("Perfumer1", "perfumer1_clean"),
    ("Perfumer2", "perfumer2_clean"),
]:
    df[dst_col] = df[src_col].apply(normalize_text) if src_col in df.columns else ""

# ── 4.5 quality filters ──
before = len(df)

df = df[df["rating_count"] >= MIN_RATING_COUNT]
log.info("After rating_count >= %d: %d rows.", MIN_RATING_COUNT, len(df))

# Nullify implausible years — keep the row, year becomes NaN.
# Year is useful generation context but not mandatory.
bad_year = df["year"].notna() & ~df["year"].between(YEAR_MIN, YEAR_MAX)
df.loc[bad_year, "year"] = float("nan")
log.info("Nulled %d implausible year values (rows kept).", int(bad_year.sum()))

# Must have >= MIN_ACCORDS main accords OR >= MIN_TOP_NOTES top notes.
has_enough_accords   = df["accords"].apply(len) >= MIN_ACCORDS
has_enough_top_notes = df["top_notes"].apply(len) >= MIN_TOP_NOTES
df = df[has_enough_accords | has_enough_top_notes]
log.info("After accord/top-note filter: %d rows.", len(df))

# Minimum distinct notes across all three tiers combined.
df["total_distinct_notes"] = df.apply(
    lambda r: len(set(r["top_notes"] + r["middle_notes"] + r["base_notes"])),
    axis=1,
)
df = df[df["total_distinct_notes"] >= MIN_TOTAL_DISTINCT_NOTES]
log.info("After distinct-note filter (>= %d): %d rows.", MIN_TOTAL_DISTINCT_NOTES, len(df))

# Placeholder: every note in the record is just an accord name repeated.
# No additional information for the generator — drop these rows.
def is_placeholder(row: pd.Series) -> bool:
    all_notes = set(row["top_notes"] + row["middle_notes"] + row["base_notes"])
    return len(all_notes) > 0 and all_notes.issubset(set(row["accords"]))

placeholder_mask = df.apply(is_placeholder, axis=1)
log.info("Removing %d placeholder note profiles.", int(placeholder_mask.sum()))
df = df[~placeholder_mask].reset_index(drop=True)

stats["after_quality_filter"] = len(df)
log.info("Quality filters complete: %d -> %d rows.", before, len(df))

# ── 4.6 near-duplicate removal ──
# Within each brand group, find pairs whose full note fingerprint
# (frozenset of top + middle + base) differs by <= MAX_NOTE_DIFF.
# Keep the higher-rated perfume in each duplicate pair.
# O(N^2) per brand — acceptable, no brand has thousands of entries.
log.info("Running near-duplicate detection (max_diff=%d notes)...", MAX_NOTE_DIFF)

fingerprints: dict[int, frozenset] = {
    idx: frozenset(
        df.at[idx, "top_notes"] +
        df.at[idx, "middle_notes"] +
        df.at[idx, "base_notes"]
    )
    for idx in df.index
}

drop_indices: set[int] = set()
for _brand, group in df.groupby("brand_clean"):
    sorted_idx = group.sort_values("rating_count", ascending=False).index.tolist()
    for i, idx_a in enumerate(sorted_idx):
        if idx_a in drop_indices:
            continue
        for idx_b in sorted_idx[i + 1:]:
            if idx_b in drop_indices:
                continue
            diff = len(fingerprints[idx_a].symmetric_difference(fingerprints[idx_b]))
            if diff <= MAX_NOTE_DIFF:
                drop_indices.add(idx_b)

before_dedup = len(df)
df = df.drop(index=list(drop_indices)).reset_index(drop=True)
log.info("Deduplication: %d -> %d rows (removed %d).", before_dedup, len(df), len(drop_indices))
stats["after_deduplication"] = len(df)

# ── 4.7 tier assignment ──
df["tier"] = df["rating_count"].apply(
    lambda rc: "gold" if rc >= GOLD_THRESHOLD else "silver"
)
tier_counts    = df["tier"].value_counts().to_dict()
total_examples = int(sum(2 if t == "gold" else 1 for t in df["tier"]))
log.info("Tiers: %s -> %d total training examples.", tier_counts, total_examples)
stats["tier_distribution"]       = tier_counts
stats["total_training_examples"] = total_examples

# ── 4.8 question type assignment (largest-remainder, interleaved) ──
# Largest-remainder guarantees the distribution is exact, not approximate.
# Interleaving prevents type clustering within any shard or session.
def build_type_sequence(n: int) -> list[str]:
    types        = [t for t, _ in QUESTION_TYPES]
    proportions  = [p for _, p in QUESTION_TYPES]
    exact        = [p * n for p in proportions]
    floored      = [int(e) for e in exact]
    deficit      = n - sum(floored)
    for i in sorted(range(len(types)), key=lambda i: -(exact[i] - floored[i]))[:deficit]:
        floored[i] += 1
    buckets = [[t] * c for t, c in zip(types, floored)]
    interleaved: list[str] = []
    while any(buckets):
        for bucket in buckets:
            if bucket:
                interleaved.append(bucket.pop(0))
    assert len(interleaved) == n, (
        f"Type sequence length mismatch: {len(interleaved)} != {n}"
    )
    return interleaved

type_seq = iter(build_type_sequence(total_examples))
df["question_types"] = [
    [next(type_seq) for _ in range(2 if t == "gold" else 1)]
    for t in df["tier"]
]

all_types = [t for slots in df["question_types"] for t in slots]
dist  = Counter(all_types)
total = sum(dist.values())
print("\nQuestion type distribution:")
for qt, count in sorted(dist.items()):
    print(f"  {qt:<25} {count:>5}  ({count / total:.1%})")

stats["question_type_distribution"] = dict(dist)

print(f"\nPreprocessing complete.")
print(f"  Final perfume count : {len(df):,}")
print(f"  Total training ex.  : {total_examples:,}")
print(f"  Gold / Silver       : {tier_counts.get('gold', 0):,} / {tier_counts.get('silver', 0):,}")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Reduces VRAM fragmentation during model loading on T4.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise EnvironmentError(
        "HF_TOKEN is empty. Check the secret value in Add-ons -> Secrets."
    )

log.info("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    trust_remote_code=True,
)

# Do NOT pass dtype here — let BitsAndBytesConfig control dtype entirely.
# Passing torch_dtype or dtype alongside quantization_config causes the new
# transformers loading pipeline to cast weights to bfloat16 before quantizing,
# which spikes VRAM usage and causes OOM on T4.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

log.info("Loading model (4-bit NF4)... first download takes 3-5 minutes.")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    # No dtype / torch_dtype argument
)
model.eval()

log.info("Model loaded.")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved  = torch.cuda.memory_reserved(i)  / 1e9
        print(f"GPU {i} ({torch.cuda.get_device_name(i)}): "
              f"allocated {allocated:.2f} GB | reserved {reserved:.2f} GB")

# Guard: if total allocated across all GPUs is under 3 GB,
# quantization silently failed — stop immediately.
total_allocated = sum(
    torch.cuda.memory_allocated(i)
    for i in range(torch.cuda.device_count())
) / 1e9
if total_allocated < 3.0:
    raise RuntimeError(
        f"Model only using {total_allocated:.2f} GB VRAM total — "
        "4-bit quantization likely failed. "
        "Check bitsandbytes version and transformers compatibility."
    )

In [ ]:
# ── Cell 6: generation utilities ──────────────────────────────────────────

def build_user_prompt(batch_records: list[dict], instructions: list[dict]) -> str:
    """Build the per-call user prompt from perfume records and generation instructions."""
    return (
        "Perfume records for this batch — generate the number of examples "
        "indicated for each, using the question types specified.\n\n"
        f"Records:\n{json.dumps(batch_records, ensure_ascii=False, indent=2)}\n\n"
        f"Instructions:\n{json.dumps(instructions, ensure_ascii=False, indent=2)}\n\n"
        "Single JSON array, raw JSON only."
    )


def record_to_dict(row: pd.Series) -> dict:
    """
    Convert a preprocessed DataFrame row into a clean dict for the prompt.

    FIX 3: year, perfumer and country use explicit index-membership checks.
    pd.Series.get(key, default) does not return the default when the key
    exists but the value is NaN — it returns NaN. Using 'key in row.index'
    followed by an explicit NaN check is unambiguous.
    """
    d: dict[str, Any] = {
        "perfume":      row["perfume_clean"],
        "brand":        row["brand_clean"],
        "gender":       row["gender_clean"],
        "top_notes":    row["top_notes"],
        "middle_notes": row["middle_notes"],
        "base_notes":   row["base_notes"],
        "accords":      row["accords"],
    }
    # Year: include only if the column exists and the value is not NaN.
    if "year" in row.index:
        year_val = row["year"]
        if year_val is not None and not pd.isna(year_val):
            d["year"] = int(year_val)
    # Perfumer: include only if the column exists and the value is non-empty.
    if "perfumer1_clean" in row.index:
        perfumer_val = row["perfumer1_clean"]
        if perfumer_val and str(perfumer_val).strip():
            d["perfumer"] = str(perfumer_val).strip()
    # Country: include only if non-empty.
    if "country_clean" in row.index:
        country_val = row["country_clean"]
        if country_val and str(country_val).strip():
            d["country"] = str(country_val).strip()
    return d


def strip_qwen_special_tokens(text: str) -> str:
    """
    FIX 2: Remove Qwen3 special tokens that survive skip_special_tokens=False.
    These appear after the closing ] of the JSON array and corrupt the regex
    fallback parser if left in place.
    """
    for tok in QWEN_SPECIAL_TOKENS:
        text = text.replace(tok, "")
    return text


def extract_json_from_response(raw: str) -> Optional[list[dict]]:
    """
    Defensively parse Qwen3's raw output into a list of {user, assistant} dicts.

    Parse strategy (in order):
      1. Strip <think>...</think> block
      2. Strip known Qwen3 special tokens  (FIX 2)
      3. Strip markdown fences
      4. json.loads() directly
      5. Regex: extract outermost [...] block and retry

    Returns None if all strategies fail.
    """
    text = raw

    # Step 1: strip thinking block
    think_end = text.find("</think>")
    if think_end != -1:
        text = text[think_end + len("</think>"):]

    # Step 2: strip Qwen3 special tokens
    text = strip_qwen_special_tokens(text).strip()

    # Step 3: strip markdown fences (``` or ```json)
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text).strip()

    # Step 4: direct parse
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass

    # Step 5: regex fallback — find the outermost [...] block
    match = re.search(r"(\[.*\])", text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(1))
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass

    return None


def validate_examples(examples: list[dict]) -> list[dict]:
    """
    Keep only examples with non-empty 'user' and 'assistant' string fields.
    Silently drops malformed entries rather than raising.
    """
    valid = []
    for ex in examples:
        if (
            isinstance(ex, dict)
            and isinstance(ex.get("user"), str)
            and isinstance(ex.get("assistant"), str)
            and ex["user"].strip()
            and ex["assistant"].strip()
        ):
            valid.append(ex)
    return valid

def generate_batch(
    batch_rows: list[pd.Series],
    debug: bool = False,
) -> tuple[Optional[list[dict]], str]:
    """
    Call Qwen3-8B for a batch of perfume rows.
    Returns (results, raw_output) always.
    results is None if parsing failed entirely.
    debug=True prints the raw output for inspection.
    """
    records = [record_to_dict(row) for row in batch_rows]
    instructions = [
        {
            "perfume_index": i,
            "examples": [{"type": qt} for qt in batch_rows[i]["question_types"]],
        }
        for i in range(len(batch_rows))
    ]

    messages = [
        {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(records, instructions)},
    ]

    # apply_chat_template returns a BatchEncoding or tensor depending on
    # transformers version. We extract the input_ids tensor explicitly
    # to avoid AttributeError in model.generate() which expects a plain tensor.
    try:
        template_output = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=True,
            thinking_budget=THINKING_BUDGET,
        )
    except (TypeError, AttributeError):
        log.warning("Thinking mode not supported. Falling back to standard generation.")
        template_output = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

    # Extract plain tensor regardless of whether we got a BatchEncoding or tensor.
    if hasattr(template_output, "input_ids"):
        input_ids = template_output.input_ids.to(model.device)
    elif isinstance(template_output, dict):
        input_ids = template_output["input_ids"].to(model.device)
    else:
        # Already a plain tensor
        input_ids = template_output.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens.
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=False)

    if debug:
        print("=" * 60)
        print("RAW OUTPUT (first 2000 chars):")
        print(raw_output[:2000])
        print("=" * 60)

    parsed = extract_json_from_response(raw_output)
    if parsed is None:
        return None, raw_output

    return validate_examples(parsed), raw_output

In [ ]:
# ── Cell 7: checkpoint and I/O utilities ──────────────────────────────────

def load_checkpoint() -> tuple[int, int, int]:
    """
    Read checkpoint.json.
    Returns (next_batch_start_idx, examples_written, failures).
    Returns (0, 0, 0) on fresh start (no checkpoint file).

    FIX 6: returns all three counters in one call so Cell 8 does not
    need a second read of the checkpoint file.
    """
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH) as f:
            data = json.load(f)
        idx      = data.get("next_batch_start_idx", 0)
        examples = data.get("examples_written", 0)
        failures = data.get("failures", 0)
        log.info(
            "Checkpoint found. Resuming from perfume index %d "
            "(%d examples written, %d failures so far).",
            idx, examples, failures,
        )
        return idx, examples, failures
    log.info("No checkpoint found. Starting fresh.")
    return 0, 0, 0


def save_checkpoint(next_idx: int, examples_written: int, failures: int) -> None:
    """Write checkpoint.json after every batch."""
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(
            {
                "next_batch_start_idx": next_idx,
                "examples_written":     examples_written,
                "failures":             failures,
            },
            f,
            indent=2,
        )


def append_examples(examples: list[dict], meta_list: list[dict]) -> None:
    """
    Append validated training examples to dataset.jsonl in messages format.
    _meta is included for auditability and is ignored by trl.SFTTrainer.
    """
    with open(DATASET_PATH, "a", encoding="utf-8") as f:
        for ex, meta in zip(examples, meta_list):
            record = {
                "messages": [
                    {"role": "system",    "content": NOSKNOWS_SYSTEM_PROMPT},
                    {"role": "user",      "content": ex["user"]},
                    {"role": "assistant", "content": ex["assistant"]},
                ],
                "_meta": meta,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def append_failure(
    batch_rows: list[pd.Series],
    raw_output: str,
    reason: str,
) -> None:
    """
    Log a failed batch to failures.jsonl.

    FIX 1: raw_output is now always the actual model output string
    (passed from generate_batch's tuple return), not an empty placeholder.
    Capped at 3000 chars to prevent huge log files.
    """
    with open(FAILURES_PATH, "a", encoding="utf-8") as f:
        record = {
            "perfumes":   [r["perfume_clean"] for r in batch_rows],
            "brands":     [r["brand_clean"]   for r in batch_rows],
            "reason":     reason,
            "raw_output": raw_output[:3000],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


print("Checkpoint and I/O utilities defined.")

In [ ]:
import torch

# Verify the model is actually on GPU, not CPU
print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

# Run a minimal generation test outside the loop
test_input = tokenizer("Hello", return_tensors="pt").to(model.device)
print("Input device:", test_input["input_ids"].device)

with torch.no_grad():
    out = model.generate(**test_input, max_new_tokens=5)
print("Test generation output:", tokenizer.decode(out[0]))

In [ ]:
# Run this once to see the real exception before starting the loop
test_batch = [df.iloc[0]]
try:
    result, raw = generate_batch(test_batch)
    print("Success:", result)
except Exception as e:
    print("Exception type:", type(e).__name__)
    print("Exception repr:", repr(e))
    import traceback
    traceback.print_exc()

In [ ]:
test_batch = [df.iloc[i] for i in range(5)]
result, raw = generate_batch(test_batch, debug=True)
print("Parsed examples:", len(result) if result else 0)
if result:
    for ex in result:
        print("\nUSER:", ex["user"])
        print("ASSISTANT:", ex["assistant"])

In [ ]:
# ── Cell 8: generation loop ───────────────────────────────────────────────
#
# Processes df in batches of BATCH_SIZE perfumes.
# After every batch:
#   - valid examples appended to dataset.jsonl
#   - checkpoint.json updated
#   - failures logged to failures.jsonl (loop continues regardless)
#
# Resume: re-run all cells. Preprocessing re-runs (CPU, ~5 min, free).
# This cell reads the checkpoint and skips already-processed batches.

rows      = [df.iloc[i] for i in range(len(df))]
n_rows    = len(rows)
batches   = [rows[i: i + BATCH_SIZE] for i in range(0, n_rows, BATCH_SIZE)]
n_batches = len(batches)

# FIX 1 + FIX 6: single call returns all three counters.
start_perfume_idx, examples_written, failure_count = load_checkpoint()
start_batch = start_perfume_idx // BATCH_SIZE

log.info(
    "Generation plan: %d perfumes | %d batches | resuming from batch %d/%d.",
    n_rows, n_batches, start_batch, n_batches,
)

session_start = time.time()

# FIX 5: tqdm progress bar — visible progress without waiting for log lines.
for batch_idx in tqdm(
    range(start_batch, n_batches),
    initial=start_batch,
    total=n_batches,
    desc="Generating",
    unit="batch",
):
    batch = batches[batch_idx]

    try:
        # FIX 1: generate_batch always returns (results, raw_output).
        results, raw_output = generate_batch(batch)
    except Exception as e:
        log.error("Batch %d raised exception: %r", batch_idx, repr(e))   
        append_failure(batch, "", f"exception: {e}")
        failure_count += 1
        save_checkpoint((batch_idx + 1) * BATCH_SIZE, examples_written, failure_count)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    if results is None or len(results) == 0:
        log.warning("Batch %d: parse failed or empty output.", batch_idx)
        # FIX 1: raw_output is the real model output, not an empty string.
        append_failure(batch, raw_output, "parse_failed")
        failure_count += 1
    else:
        expected_total = sum(len(row["question_types"]) for row in batch)
        if len(results) != expected_total:
            log.warning(
                "Batch %d: expected %d examples, got %d. Saving what is available.",
                batch_idx, expected_total, len(results),
            )

        # Assign metadata by walking perfumes and their question slots in order.
        # Qwen is instructed to produce examples in perfume_index order.
        # StopIteration on fewer results than expected stops assignment cleanly.
        meta_list:     list[dict] = []
        valid_results: list[dict] = []
        result_iter = iter(results)

        for row in batch:
            for qt in row["question_types"]:
                try:
                    ex = next(result_iter)
                    valid_results.append(ex)
                    meta_list.append({
                        "perfume":       row["perfume_clean"],
                        "brand":         row["brand_clean"],
                        "question_type": qt,
                        "tier":          row["tier"],
                    })
                except StopIteration:
                    break

        append_examples(valid_results, meta_list)
        examples_written += len(valid_results)

    # Checkpoint after every batch — safe against mid-session kill.
    save_checkpoint((batch_idx + 1) * BATCH_SIZE, examples_written, failure_count)

    # Detailed progress log every 10 batches with rolling ETA.
    if (batch_idx - start_batch) % 10 == 0:
        elapsed   = time.time() - session_start
        remaining = n_batches - batch_idx - 1
        avg_batch = elapsed / max(batch_idx - start_batch + 1, 1)
        eta_hours = (remaining * avg_batch) / 3600
        log.info(
            "Batch %d/%d | examples: %d | failures: %d | avg: %.1fs/batch | ETA: %.1fh",
            batch_idx + 1, n_batches,
            examples_written, failure_count,
            avg_batch, eta_hours,
        )

    # Free KV cache between batches to prevent VRAM fragmentation over long runs.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

log.info("Generation loop complete for this session.")

In [ ]:
# ── Cell 9: report ────────────────────────────────────────────────────────
#
# FIX 6: reads all values from disk rather than relying on in-memory variables.
# Safe to run standalone at any point mid-session to check progress.

actual_examples = 0
if DATASET_PATH.exists():
    with open(DATASET_PATH, encoding="utf-8") as f:
        actual_examples = sum(1 for _ in f)

actual_failures = 0
if FAILURES_PATH.exists():
    with open(FAILURES_PATH, encoding="utf-8") as f:
        actual_failures = sum(1 for _ in f)

ckpt_data: dict = {}
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH) as f:
        ckpt_data = json.load(f)

# Merge preprocessing stats (available when Cell 4 ran in this session)
# with any persisted report from a previous session.
report_data: dict = {}
if REPORT_PATH.exists():
    with open(REPORT_PATH) as f:
        report_data = json.load(f)
if "stats" in dir():
    report_data.update(stats)  # stats built in Cell 4

batches_processed = ckpt_data.get("next_batch_start_idx", 0) // max(BATCH_SIZE, 1)
success_rate = (
    f"{(1 - actual_failures / max(batches_processed, 1)) * 100:.1f}%"
    if batches_processed > 0 else "n/a"
)

report_data["examples_generated"] = actual_examples
report_data["batch_failures"]     = actual_failures
report_data["success_rate"]       = success_rate
report_data["checkpoint"]         = ckpt_data

with open(REPORT_PATH, "w") as f:
    json.dump(report_data, f, indent=2)

print("\n" + "=" * 62)
print("GENERATION REPORT")
print("=" * 62)
for label, key in [
    ("Input rows (raw CSV)",     "input_rows"),
    ("After structural drop",    "after_structural_drop"),
    ("After quality filter",     "after_quality_filter"),
    ("After deduplication",      "after_deduplication"),
    ("Target training examples", "total_training_examples"),
]:
    if key in report_data:
        print(f"  {label:<30}: {report_data[key]:>8,}")
if "tier_distribution" in report_data:
    td = report_data["tier_distribution"]
    print(f"  {'Gold / Silver':<30}: {td.get('gold', 0):>5,} / {td.get('silver', 0):,}")
print(f"  {'Examples written so far':<30}: {actual_examples:>8,}")
print(f"  {'Failed batches so far':<30}: {actual_failures:>8,}")
print(f"  {'Batch success rate':<30}: {success_rate}")
if ckpt_data:
    nxt = ckpt_data.get('next_batch_start_idx', 0)
    print(f"  {'Next perfume index (resume)':<30}: {nxt:>8,}")
print("=" * 62)
print(f"\ndataset.jsonl  -> {DATASET_PATH}")
print(f"failures.jsonl -> {FAILURES_PATH}")
print(f"report         -> {REPORT_PATH}")
print("\nDownload from the Output tab when done.")
print("The _meta field in each JSONL line is ignored by trl.SFTTrainer automatically.")